# Okay is this working? Hello? Guys? HELP ME?

In [ ]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split    
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report 
import requests
from bs4 import BeautifulSoup
import re
import time
from tqdm.notebook import tqdm
from urllib.parse import urlparse
# Load the dataset
data = pd.read_csv('datasets/FakeNewsNet.csv')
#gitignore!!!

In [2]:
df = pd.DataFrame(data)
print(df.head())
print(df.info())


                                               title  \
0  Kandi Burruss Explodes Over Rape Accusation on...   
1  People's Choice Awards 2018: The best red carp...   
2  Sophia Bush Sends Sweet Birthday Message to 'O...   
3  Colombian singer Maluma sparks rumours of inap...   
4  Gossip Girl 10 Years Later: How Upper East Sid...   

                                            news_url        source_domain  \
0  http://toofab.com/2017/05/08/real-housewives-a...           toofab.com   
1  https://www.today.com/style/see-people-s-choic...        www.today.com   
2  https://www.etonline.com/news/220806_sophia_bu...     www.etonline.com   
3  https://www.dailymail.co.uk/news/article-33655...  www.dailymail.co.uk   
4  https://www.zerchoo.com/entertainment/gossip-g...      www.zerchoo.com   

   tweet_num  real  
0         42     1  
1          0     1  
2         63     1  
3         20     1  
4         38     1  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23196 entries, 0 to 2319

In [3]:
# not nulls
print(df.isnull().sum())

title              0
news_url         330
source_domain    330
tweet_num          0
real               0
dtype: int64


# Prepare for scraping

In [4]:
print(df.columns.tolist())

URL_COL = "news_url"
assert URL_COL in df.columns, f"Column '{URL_COL}' not found in CSV"

['title', 'news_url', 'source_domain', 'tweet_num', 'real']


##

In [ ]:
# Beautiful Soup Template https://www.crummy.com/software/BeautifulSoup/bs4/doc/#quick-start

# User-Agent header to mimic a real browser and avoid blocking, can be expanded to rotate or use proxies if needed. Whatever man.
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )
}

# Helper functions for text cleaning and word extraction
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def extract_words(text):
    if not isinstance(text, str):
        return []
    return re.findall(r"\b[a-zA-Z]+\b", text.lower())





#==================================== Main scraping function with error handling and smart content extraction===================================================
def scrape_website(url, timeout=10):
    if pd.isna(url) or not isinstance(url, str) or not url.strip(): #shits busted
        return "", "", 0, "missing_url" # make sure to log that we are missing urls
    try:
        response = requests.get(url, headers=HEADERS, timeout=timeout)
        response.raise_for_status() # will raise an HTTPError for bad responses (4xx and 5xx) "not found" and "server error" and all that jizz
        soup = BeautifulSoup(response.text, "html.parser") # gets rid of all the shit we dont need and makes it easier to extract the text



#======================================= ---JUNK WORDS REMOVAL--- ========================================================
        for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside"]):
            tag.decompose() # execute it



        # Smart Article Extraction: Try common tags first, then fallback to body or whole text
        content = soup.find("article")
        if content is None:
            content = soup.find("main")
        if content is None:
            content = soup.body
        if content is None:
            content = soup

        text = content.get_text(separator=" ")
        text = clean_text(text) # helper frunction from above

        # hehe frunction

        words = extract_words(text)
        words_joined = " ".join(words)

        return text, words_joined, len(words), "success"

    # fun fact not havbing these crashes your entire trace and you lose all your progress!
    # LOG
    # YOUR
    # SHIT
    except requests.exceptions.RequestException as e:
        return "", "", 0, f"request_error: {str(e)}"
    except Exception as e:
        return "", "", 0, f"parse_error: {str(e)}"

# URL Normalization
- Beautsoup dies if there is no https:// so i found a libnrtary that does that

In [ ]:
# sanity check
from url_normalize import url_normalize

test_urls = [
    "www.cnn.com",
    "cnn.com",
    "foxnews.com/article/123",
    "https://www.nytimes.com",
    "http://www.bbc.com/news",
    "//reuters.com/world",
    "www.example.com/test",
    "https://https://abcnews.go.com",
    "www.wsj.com/articles/something"
]

for u in test_urls:
    print(u, " -> ", url_normalize(u))

# HYPE

www.cnn.com  ->  https://www.cnn.com/
cnn.com  ->  https://cnn.com/
foxnews.com/article/123  ->  https://foxnews.com/article/123
https://www.nytimes.com  ->  https://www.nytimes.com/
http://www.bbc.com/news  ->  http://www.bbc.com/news
//reuters.com/world  ->  https://reuters.com/world
www.example.com/test  ->  https://www.example.com/test
https://https://abcnews.go.com  ->  https://https/abcnews.go.com
www.wsj.com/articles/something  ->  https://www.wsj.com/articles/something


## Scrape Once.

In [7]:

test_url = df.loc[0, URL_COL]
print("Testing:", test_url)

text, words, count, status = scrape_website(test_url)
print("Status:", status)
print("Word count:", count)
print(text[:500])

# cool

Testing: http://toofab.com/2017/05/08/real-housewives-atlanta-kandi-burruss-rape-phaedra-parks-porsha-williams/
Status: success
Word count: 13
Savannah Guthrie Returns to ‘Today’ Show for First Time Since Mom Went Missing


# John Ryan Scrape

In [10]:
# Fragnum Opus: The Freat Frcrape

# ========================================================== Settings===========================================================
OUTPUT_FILE = "FakeNewsNet_scraped.csv"
CHECKPOINT_EVERY = 100 # save progress every N rows to avoid losing everything if something goes wrong
SLEEP_SECONDS = 0.8    # be kiond to servers, adjust as needed, lowering value might speed up but increases risk of being blocked
START_AT = 0           # change this if resuming manually

# ========================================================== Main Loop ============================================================

# output columns
for col in ["scraped_text", "scraped_words", "scraped_word_count", "scrape_status"]:
    if col not in df.columns:
        df[col] = None

# URL Normalization: Clean and standardize URLs to improve scraping success rates, handles missing schemes, trailing slashes, etc.
df["fixed_url"] = df[URL_COL].apply(
    lambda u: url_normalize(u) if isinstance(u, str) and u.strip() else None
)
df[[URL_COL, "fixed_url"]].head(20)


# really cool progress bar with tqdm i found, it shows progress and estimated time remaining, super helpful for long scrapes
for i in tqdm(range(START_AT, len(df)), desc="Scraping websites"):

    # Skip rows already completed- super useful for resuming after crashes or if you want to run in batches
    if pd.notna(df.at[i, "scrape_status"]) and str(df.at[i, "scrape_status"]).strip() != "":
        continue


# Mainline BSOUP scraping logic, call our helper function and store results in the dataframe
    url = df.at[i, "fixed_url"]
    text, words, count, status = scrape_website(url)

    df.at[i, "scraped_text"] = text
    df.at[i, "scraped_words"] = words
    df.at[i, "scraped_word_count"] = count
    df.at[i, "scrape_status"] = status

    # checkpointing: Save progress every N rows to avoid losing everything if something goes wrong, super important for long scrapes
    if (i + 1) % CHECKPOINT_EVERY == 0:
        df.to_csv(OUTPUT_FILE, index=False)
        print("Checkpoint Saved.")
    
    # john status update for each row, helps track progress and debug issues with specific URLs
    print(f"Row {i+1}/{len(df)}: URL='{url}' Status='{status}' Word Count={count}")

    time.sleep(SLEEP_SECONDS)

# Final save
df.to_csv(OUTPUT_FILE, index=False)
print(f"Yo Chud. Im Done. Saved to {OUTPUT_FILE}")

Scraping websites:   0%|          | 0/23196 [00:00<?, ?it/s]

Row 1/23196: URL='http://toofab.com/2017/05/08/real-housewives-atlanta-kandi-burruss-rape-phaedra-parks-porsha-williams/' Status='success' Word Count=13
Row 2/23196: URL='https://www.today.com/style/see-people-s-choice-awards-red-carpet-looks-t141832' Status='success' Word Count=784
Row 3/23196: URL='https://www.etonline.com/news/220806_sophia_bush_sends_sweet_birthday_message_to_one_tree_hill_co_star_hilarie_burton_breyton_4eva' Status='success' Word Count=412
Row 4/23196: URL='https://www.dailymail.co.uk/news/article-3365543/Colombian-music-star-sparks-rumours-inappropriate-relationship-yoga-instructor-AUNT-posting-sexually-suggestive-photographs-pair.html' Status='success' Word Count=4292
Row 5/23196: URL='https://www.zerchoo.com/entertainment/gossip-girl-10-years-later-how-upper-east-siders-shocked-the-world-changed-pop-culture-forever/' Status='request_error: 404 Client Error: Not Found for url: https://zerchoo.com/entertainment/gossip-girl-10-years-later-how-upper-east-siders-sho

C:\Users\ryan1\AppData\Local\Temp\ipykernel_28380\2547154557.py:29: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(response.text, "html.parser")


Row 1403/23196: URL='http://clerk.house.gov/evs/2009/roll884.xml' Status='success' Word Count=1029
Row 1404/23196: URL='https://www.cnn.com/2016/09/12/entertainment/ryan-lochte-dancing-with-the-stars/index.html' Status='success' Word Count=1113
Row 1405/23196: URL='https://www.distractify.com/trending/2017/08/03/bQzp0/husbands-curvy-wife-instagram' Status='success' Word Count=456
Row 1406/23196: URL='https://deadline.com/2018/01/greys-anatomy-spinoff-title-1202269247/' Status='success' Word Count=355
Row 1407/23196: URL='https://www.cheatsheet.com/entertainment/kylie-jenner-keeping-fans-guessing-pregnancy.html/' Status='success' Word Count=58
Row 1408/23196: URL='https://www.biography.com/people/jerry-seinfeld-9542107' Status='success' Word Count=2207
Row 1409/23196: URL='https://www.indy100.com/article/chrissy-teigen-childbirth-mother-instagram-ivf-pregnancy-body-8361761' Status='success' Word Count=15
Row 1410/23196: URL='https://www.etonline.com/watch-jennifer-lawrence-hilariously-t

KeyboardInterrupt: 

In [ ]:
# Cell 8: Save output
df.to_csv("FakeNewsNet_scraped.csv", index=False)
print("Saved to FakeNewsNet_scraped.csv")
# Cell 9: Quick check
df[["news_url", "scrape_status", "scraped_word_count"]].head(20)

Saved to FakeNewsNet_scraped.csv


,news_url,scrape_status,scraped_word_count
0,http://toofab.com/2017/05/08/real-housewives-a...,success,13
1,https://www.today.com/style/see-people-s-choic...,success,784
2,https://www.etonline.com/news/220806_sophia_bu...,success,412
3,https://www.dailymail.co.uk/news/article-33655...,success,4272
4,https://www.zerchoo.com/entertainment/gossip-g...,request_error: 404 Client Error: Not Found for...,0
5,www.intouchweekly.com/posts/gwen-stefani-dumpe...,request_error: Invalid URL 'www.intouchweekly....,0
6,https://yournewswire.com/broward-county-sherif...,request_error: 404 Client Error: Not Found for...,0
7,www.etonline.com/news/214798_amber_rose_shuts_...,request_error: Invalid URL 'www.etonline.com/n...,0
8,https://www.aol.com/article/entertainment/2018...,request_error: 404 Client Error: Not Found for...,0
9,https://www.98online.com/2018/05/02/katharine-...,request_error: 404 Client Error: Not Found for...,0
